New Patient

⬇

Embedding

⬇

Compare With Past Cases

⬇

Rank Similar Cases

⬇

Return Top Matches

In [5]:
import pandas as pd
import numpy as np
import json
import os


# LOAD DATASET 
FILE_PATH = r"D:\chiselon\Week 0\Week_0_Prep_Week_Ssample Data_clinic_cases.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError("CSV file not found at given path.")

df = pd.read_csv(FILE_PATH)

print("Dataset Loaded Successfully!")
print("Total Cases:", len(df))
print("Columns:", df.columns.tolist())

# TEXT PREPROCESSING
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower().strip()


def combine_text(symptoms, doctor_notes):
    symptoms_clean = preprocess_text(symptoms)
    notes_clean = preprocess_text(doctor_notes)
    return symptoms_clean + " " + notes_clean

# EMBEDDING FUNCTION 
def generate_embedding(text):
    
    text = preprocess_text(text)

    length_feature = len(text)
    word_count = len(text.split())
    char_sum_feature = sum(ord(c) for c in text) % 1000
    vowel_count = sum(c in "aeiou" for c in text)

    return np.array([length_feature, word_count, char_sum_feature, vowel_count])

# COSINE SIMILARITY

def compute_cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0

    return dot_product / (norm_vec1 * norm_vec2)
# PRECOMPUTE CASE EMBEDDINGS (IMPORTANT)

case_embeddings = []

for index, row in df.iterrows():
    combined = combine_text(row['symptoms'], row['doctor_notes'])
    embedding = generate_embedding(combined)

    case_embeddings.append({
        "case_id": row['case_id'],
        "embedding": embedding,
        "treatment": row.get('treatment', "NA"),
        "outcome": row.get('outcome', "NA")
    })

# RETRIEVE SIMILAR CASES
def retrieve_similar_cases(new_embedding, top_n=5):

    similarities = []

    for case in case_embeddings:
        score = compute_cosine_similarity(new_embedding, case["embedding"])

        similarities.append({
            "case_id": case["case_id"],
            "similarity_score": round(float(score), 4),
            "treatment": case["treatment"],
            "outcome": case["outcome"]
        })

    # Sort descending
    similarities = sorted(
        similarities,
        key=lambda x: x['similarity_score'],
        reverse=True
    )

    return similarities[:top_n]

# INSIGHT GENERATION
def generate_case_insight(similar_cases):

    treatments = []
    outcomes = []

    for case in similar_cases:
        treatments.append(case["treatment"])
        outcomes.append(case["outcome"])

    if len(treatments) == 0:
        return {"message": "No similar cases found."}

    most_common_treatment = max(set(treatments), key=treatments.count)

    recovery_count = outcomes.count("Improved")
    recovery_percentage = (recovery_count / len(outcomes)) * 100

    insight = {
        "most_common_treatment": most_common_treatment,
        "recovery_trend": f"{round(recovery_percentage,2)}% cases showed improvement",
        "confidence_reason": f"Based on {len(similar_cases)} structurally similar historical cases"
    }

    return insight

# COMPLETE PIPELINE
def similarity_engine_pipeline(symptoms, doctor_notes):

    combined_text = combine_text(symptoms, doctor_notes)
    new_embedding = generate_embedding(combined_text)

    similar_cases = retrieve_similar_cases(new_embedding, top_n=5)

    insight = generate_case_insight(similar_cases)

    output = {
        "similar_cases": similar_cases,
        "insight_summary": insight
    }

    return output

# DEMONSTRATION
if __name__ == "__main__":

    new_symptoms = "Severe chest pain radiating to arm"
    new_notes = "Patient sweating and breathless. Known hypertension."

    result = similarity_engine_pipeline(new_symptoms, new_notes)

    print("\n=== CCMS Similarity Engine Output (Day-3 Version) ===\n")
    print(json.dumps(result, indent=4))

Dataset Loaded Successfully!
Total Cases: 10
Columns: ['case_id', 'clinic_id', 'symptoms', 'duration_days', 'doctor_notes', 'diagnosis', 'treatment', 'outcome', 'recovery_days', 'patient.age', 'patient.gender']

=== CCMS Similarity Engine Output (Day-3 Version) ===

{
    "similar_cases": [
        {
            "case_id": "CASE_003",
            "similarity_score": 1.0,
            "treatment": "Chemical peel + sunscreen",
            "outcome": "Partially Improved"
        },
        {
            "case_id": "CASE_002",
            "similarity_score": 0.9962,
            "treatment": "Topical steroid + antihistamine",
            "outcome": "Improved"
        },
        {
            "case_id": "CASE_005",
            "similarity_score": 0.9951,
            "treatment": "Botulinum toxin evaluation",
            "outcome": "Referred"
        },
        {
            "case_id": "CASE_008",
            "similarity_score": 0.9942,
            "treatment": "Laser evaluation",
            

In [7]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# LOAD DATASET
FILE_PATH = r"D:\chiselon\Week 0\Week_0_Prep_Week_Ssample Data_clinic_cases.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError("CSV file not found at given path.")

df = pd.read_csv(FILE_PATH)

print("Dataset Loaded Successfully!")
print("Total Cases:", len(df))

# TEXT PREPROCESSING
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower().strip()


def combine_text(symptoms, doctor_notes):
    return preprocess_text(symptoms) + " " + preprocess_text(doctor_notes)

# ADVANCED EMBEDDING
def generate_embedding(text):
    text = preprocess_text(text)

    length_feature = len(text)
    word_count = len(text.split())
    char_sum_feature = sum(ord(c) for c in text) % 1000
    vowel_count = sum(c in "aeiou" for c in text)

    return np.array([length_feature, word_count, char_sum_feature, vowel_count])

# PRECOMPUTE ALL CASE EMBEDDINGS (VECTORIZED)
case_ids = []
treatments = []
outcomes = []
diagnoses = []
embedding_matrix = []

for _, row in df.iterrows():

    combined = combine_text(row['symptoms'], row['doctor_notes'])
    embedding = generate_embedding(combined)

    case_ids.append(row['case_id'])
    treatments.append(row.get('treatment', "NA"))
    outcomes.append(row.get('outcome', "NA"))
    diagnoses.append(row.get('diagnosis', "NA"))
    embedding_matrix.append(embedding)

embedding_matrix = np.array(embedding_matrix)

# FAST COSINE SIMILARITY (VECTOR FORM)
def compute_similarity_vectorized(new_embedding):
    
    dot_products = np.dot(embedding_matrix, new_embedding)
    
    norm_cases = np.linalg.norm(embedding_matrix, axis=1)
    norm_new = np.linalg.norm(new_embedding)

    similarities = dot_products / (norm_cases * norm_new + 1e-8)

    return similarities

# CONFIDENCE CLASSIFIER
def classify_confidence(avg_similarity):

    if avg_similarity > 0.85:
        return "Very High"
    elif avg_similarity > 0.70:
        return "High"
    elif avg_similarity > 0.55:
        return "Moderate"
    else:
        return "Low"

# RETRIEVAL ENGINE
def retrieve_similar_cases(symptoms, doctor_notes, top_n=5, threshold=0.55):

    combined = combine_text(symptoms, doctor_notes)
    new_embedding = generate_embedding(combined)

    similarity_scores = compute_similarity_vectorized(new_embedding)

    results = []

    for i in range(len(similarity_scores)):

        if similarity_scores[i] >= threshold:

            results.append({
                "case_id": case_ids[i],
                "diagnosis": diagnoses[i],
                "treatment": treatments[i],
                "outcome": outcomes[i],
                "similarity_score": round(float(similarity_scores[i]), 4)
            })

    # Sort descending
    results = sorted(results, key=lambda x: x["similarity_score"], reverse=True)

    top_results = results[:top_n]

    return top_results

# INSIGHT + DIAGNOSIS SUGGESTION
def generate_advanced_insight(similar_cases):

    if len(similar_cases) == 0:
        return {"message": "No clinically similar cases found."}

    treatments = [case["treatment"] for case in similar_cases]
    diagnoses_list = [case["diagnosis"] for case in similar_cases]
    similarities = [case["similarity_score"] for case in similar_cases]
    outcomes = [case["outcome"] for case in similar_cases]

    most_common_treatment = max(set(treatments), key=treatments.count)
    suggested_diagnosis = max(set(diagnoses_list), key=diagnoses_list.count)

    recovery_count = outcomes.count("Improved")
    recovery_percentage = (recovery_count / len(outcomes)) * 100

    avg_similarity = np.mean(similarities)
    confidence_level = classify_confidence(avg_similarity)

    return {
        "suggested_diagnosis": suggested_diagnosis,
        "most_common_treatment": most_common_treatment,
        "recovery_trend": f"{round(recovery_percentage,2)}% showed improvement",
        "average_similarity": round(float(avg_similarity), 4),
        "confidence_level": confidence_level,
        "based_on_cases": len(similar_cases)
    }

# COMPLETE ADVANCED PIPELINE
def advanced_similarity_engine(symptoms, doctor_notes):

    similar_cases = retrieve_similar_cases(symptoms, doctor_notes)

    insight = generate_advanced_insight(similar_cases)

    output = {
        "query_timestamp": str(datetime.now()),
        "similar_cases": similar_cases,
        "insight_summary": insight
    }

    return output

# DEMO
if __name__ == "__main__":

    new_symptoms = "Severe chest pain radiating to left arm"
    new_notes = "Patient sweating and breathless. Known hypertension."

    result = advanced_similarity_engine(new_symptoms, new_notes)

    print("\n=== ADVANCED DAY-3 SIMILARITY ENGINE OUTPUT ===\n")
    print(json.dumps(result, indent=4))

Dataset Loaded Successfully!
Total Cases: 10

=== ADVANCED DAY-3 SIMILARITY ENGINE OUTPUT ===

{
    "query_timestamp": "2026-02-20 11:25:12.890215",
    "similar_cases": [
        {
            "case_id": "CASE_004",
            "diagnosis": "Vitiligo",
            "treatment": "Topical tacrolimus",
            "outcome": "No Change",
            "similarity_score": 1.0
        },
        {
            "case_id": "CASE_008",
            "diagnosis": "Solar Lentigines",
            "treatment": "Laser evaluation",
            "outcome": "Referred",
            "similarity_score": 1.0
        },
        {
            "case_id": "CASE_002",
            "diagnosis": "Contact Dermatitis",
            "treatment": "Topical steroid + antihistamine",
            "outcome": "Improved",
            "similarity_score": 0.9999
        },
        {
            "case_id": "CASE_007",
            "diagnosis": "Seborrheic Dermatitis",
            "treatment": "Medicated shampoo",
            "outcome